In [1]:
import tvm
from tvm import relax
from tvm.relax.frontend.torch import from_exported_program

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import torchvision.models as models
from torchvision.io import read_image
from torch.export import export

import onnx
import os

# Model

just a simple model using some softmax, designed to test the Meta Scheduling feature.

In [2]:
class SoftmaxMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(128, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)

    def forward(self, x):
        x = self.fc1(x)
        x = F.softmax(x, dim=-1)

        x = self.fc2(x)
        x = F.softmax(x, dim=-1)

        x = self.fc3(x)
        x = F.softmax(x, dim=-1)

        return x
    
torch_model = SoftmaxMLP().eval()

IRModule

In [3]:
# Give an example argument to torch.export
example_args = (torch.randn(1, 128, dtype=torch.float32),)

# Convert the model to IRModule
with torch.no_grad():
    exported_program = export(torch_model, example_args)
    mod = from_exported_program(exported_program, keep_params_as_input=True)

mod, params = relax.frontend.detach_params(mod)
mod.show()

/usr/lib/python3/dist-packages/z3/z3core.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Optimize model

In [5]:
TOTAL_TRIALS = 512
MAX_TRIALS_PER_TASK = 16 
target = tvm.target.Target({
    "kind": "llvm",
    "mtriple": "riscv64-linux-gnu",
    "mcpu": "generic-rv64",
    "mattr": ["+64bit", "+m", "+a", "+f", "+d", "+c", "+v"],
    "num-cores": 8,
})
work_dir = "tuning_logs"

mod = relax.get_pipeline(
    "static_shape_tuning",
    target=target,
    work_dir=work_dir,
    total_trials=TOTAL_TRIALS,
    max_trials_per_task=MAX_TRIALS_PER_TASK,
)(mod)

mod["main"].show()

2026-03-25 16:31:12 [INFO] Logging directory: tuning_logs/logs


2026-03-25 16:31:13 [INFO] LocalBuilder: max_workers = 8
2026-03-25 16:31:14 [INFO] LocalRunner: max_workers = 1
2026-03-25 16:31:15 [INFO] [task_scheduler.cc:168] Initializing Task #0: "softmax2"
2026-03-25 16:31:15 [INFO] [task_scheduler.cc:168] Initializing Task #1: "fused_matmul2_add2"
2026-03-25 16:31:15 [INFO] [task_scheduler.cc:168] Initializing Task #2: "fused_matmul1_add1"
2026-03-25 16:31:15 [INFO] [task_scheduler.cc:168] Initializing Task #3: "softmax"
2026-03-25 16:31:15 [INFO] [task_scheduler.cc:168] Initializing Task #4: "softmax1"
2026-03-25 16:31:16 [INFO] [task_scheduler.cc:168] Initializing Task #5: "fused_matmul_add"
2026-03-25 16:31:16 [INFO] [task_scheduler.cc:168] Initializing Task #6: "transpose2"
2026-03-25 16:31:16 [INFO] [task_scheduler.cc:168] Initializing Task #7: "transpose1"
2026-03-25 16:31:16 [INFO] [task_scheduler.cc:168] Initializing Task #8: "transpose"


,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,softmax2,256,1,N/A,N/A,N/A,0,
1,fused_matmul2_add2,16448,1,N/A,N/A,N/A,0,
2,fused_matmul1_add1,65664,1,N/A,N/A,N/A,0,
3,softmax,1024,1,N/A,N/A,N/A,0,
4,softmax1,512,1,N/A,N/A,N/A,0,
5,fused_matmul_add,65792,1,N/A,N/A,N/A,0,
6,transpose2,1,1,N/A,N/A,N/A,0,
7,transpose1,1,1,N/A,N/A,N/A,0,
8,transpose,1,1,N/A,N/A,N/A,0,


2026-03-25 16:31:17 [DEBUG] [task_scheduler.cc:327] 
 ID |               Name |  FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
------------------------------------------------------------------------------------------------------------------
  0 |           softmax2 |   256 |      1 |            N/A |          N/A |                   N/A |      0 |      
  1 | fused_matmul2_add2 | 16448 |      1 |            N/A |          N/A |                   N/A |      0 |      
  2 | fused_matmul1_add1 | 65664 |      1 |            N/A |          N/A |                   N/A |      0 |      
  3 |            softmax |  1024 |      1 |            N/A |          N/A |                   N/A |      0 |      
  4 |           softmax1 |   512 |      1 |            N/A |          N/A |                   N/A |      0 |      
  5 |   fused_matmul_add | 65792 |      1 |            N/A |          N/A |                   N/A |      0 |      
  6 |         transpose2 | 

[16:31:33] /home/leobrasileo/Desktop/UBA/Tesis/TVM-RVV_optimized_operators/tvm/src/script/printer/tir/expr.cc:74: Warning: Didn't find variable definition for: v_k_o
[16:31:33] /home/leobrasileo/Desktop/UBA/Tesis/TVM-RVV_optimized_operators/tvm/src/script/printer/tir/expr.cc:74: Warning: Didn't find variable definition for: v_k_o
[16:31:33] /home/leobrasileo/Desktop/UBA/Tesis/TVM-RVV_optimized_operators/tvm/src/script/printer/ir/./../utils.h:45: Warning: TVMScript printer falls back to the basic address printer with the error:
Variable is not defined in the environment: v_k_o
[16:31:33] /home/leobrasileo/Desktop/UBA/Tesis/TVM-RVV_optimized_operators/tvm/src/script/printer/ir/./../utils.h:45: Warning: TVMScript printer falls back to the basic address printer with the error:
Variable is not defined in the environment: v_k_o
[16:31:33] /home/leobrasileo/Desktop/UBA/Tesis/TVM-RVV_optimized_operators/tvm/src/meta_schedule/postproc/rewrite_tensorize.cc:47: Warning: Tensorize failed with erro

...

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,softmax2,256,1,N/A,N/A,N/A,16,Y
1,fused_matmul2_add2,16448,1,N/A,N/A,N/A,16,Y
2,fused_matmul1_add1,65664,1,N/A,N/A,N/A,16,Y
3,softmax,1024,1,N/A,N/A,N/A,16,Y
4,softmax1,512,1,N/A,N/A,N/A,16,Y
5,fused_matmul_add,65792,1,N/A,N/A,N/A,16,Y
6,transpose2,1,1,N/A,N/A,N/A,1,Y
7,transpose1,1,1,N/A,N/A,N/A,1,
8,transpose,1,1,N/A,N/A,N/A,1,Y



Total trials: 0
Total latency (us): 0

2026-03-25 16:33:29 [DEBUG] [task_scheduler.cc:327] 
 ID |               Name |  FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
------------------------------------------------------------------------------------------------------------------
  0 |           softmax2 |   256 |      1 |            N/A |          N/A |                   N/A |     16 |    Y 
  1 | fused_matmul2_add2 | 16448 |      1 |            N/A |          N/A |                   N/A |     16 |    Y 
  2 | fused_matmul1_add1 | 65664 |      1 |            N/A |          N/A |                   N/A |     16 |    Y 
  3 |            softmax |  1024 |      1 |            N/A |          N/A |                   N/A |     16 |    Y 
  4 |           softmax1 |   512 |      1 |            N/A |          N/A |                   N/A |     16 |    Y 
  5 |   fused_matmul_add | 65792 |      1 |            N/A |          N/A |                   N/A |   

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,softmax2,256,1,N/A,N/A,N/A,16,Y
1,fused_matmul2_add2,16448,1,N/A,N/A,N/A,16,Y
2,fused_matmul1_add1,65664,1,N/A,N/A,N/A,16,Y
3,softmax,1024,1,N/A,N/A,N/A,16,Y
4,softmax1,512,1,N/A,N/A,N/A,16,Y
5,fused_matmul_add,65792,1,N/A,N/A,N/A,16,Y
6,transpose2,1,1,N/A,N/A,N/A,1,Y
7,transpose1,1,1,N/A,N/A,N/A,1,Y
8,transpose,1,1,N/A,N/A,N/A,1,Y


2026-03-25 16:33:29 [DEBUG] [task_scheduler.cc:327] 
 ID |               Name |  FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
------------------------------------------------------------------------------------------------------------------
  0 |           softmax2 |   256 |      1 |            N/A |          N/A |                   N/A |     16 |    Y 
  1 | fused_matmul2_add2 | 16448 |      1 |            N/A |          N/A |                   N/A |     16 |    Y 
  2 | fused_matmul1_add1 | 65664 |      1 |            N/A |          N/A |                   N/A |     16 |    Y 
  3 |            softmax |  1024 |      1 |            N/A |          N/A |                   N/A |     16 |    Y 
  4 |           softmax1 |   512 |      1 |            N/A |          N/A |                   N/A |     16 |    Y 
  5 |   fused_matmul_add | 65792 |      1 |            N/A |          N/A |                   N/A |     16 |    Y 
  6 |         transpose2 | 

[16:33:29] /home/leobrasileo/Desktop/UBA/Tesis/TVM-RVV_optimized_operators/tvm/src/relax/transform/meta_schedule.cc:92: Warning: Creating JSONDatabase. Workload at: tuning_logs/database_workload.json, Tuning records at: tuning_logs/database_tuning_record.json


In [ ]:
with target:
    mod = tvm.s_tir.transform.DefaultGPUSchedule()(mod)
ex = tvm.compile(mod, target=target)